# Task 3.6 — Huấn luyện RNN (Colab GPU)

Notebook này chạy trên **Google Colab với GPU T4**.

## Trước khi chạy
1. Vào **Runtime → Change runtime type → T4 GPU**
2. Đảm bảo Google Drive đã sync xong folder `heart-murmur/`
3. Chạy từng cell theo thứ tự từ trên xuống

## Cấu trúc folder trên Google Drive
```
MyDrive/heart-murmur/
├── data/
│   ├── processed/
│   │   ├── spectrograms/   (3163 file .npy)
│   │   └── labels/         (3163 file .npy)
│   └── metadata/
│       ├── patients.csv
│       ├── recordings.csv
│       └── cv_splits.json
└── src/
    ├── __init__.py
    ├── data/
    │   ├── __init__.py
    │   └── loader.py
    └── models/
        ├── __init__.py
        └── rnn.py
```

## Output (tải về máy sau khi train xong)
```
MyDrive/heart-murmur/
├── models/rnn/fold{0-4}_best.pt
└── experiments/logs/fold{0-4}_loss.csv
```

## Cell 1 — Kiểm tra GPU và mount Drive

In [2]:
# Kiểm tra GPU
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không có GPU — vào Runtime → Change runtime type → T4 GPU')

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Đường dẫn project trên Drive
PROJECT_ROOT = Path('/content/drive/MyDrive/heart-murmur')

# Thêm vào Python path để import src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Kiểm tra
print(f'Project root exists : {PROJECT_ROOT.exists()}')
print(f'src/ exists         : {(PROJECT_ROOT / "src").exists()}')
print(f'spectrograms/ exists: {(PROJECT_ROOT / "data/processed/spectrograms").exists()}')
print(f'labels/ exists      : {(PROJECT_ROOT / "data/processed/labels").exists()}')

# Đếm file
n_spec  = len(list((PROJECT_ROOT / 'data/processed/spectrograms').glob('*.npy')))
n_label = len(list((PROJECT_ROOT / 'data/processed/labels').glob('*.npy')))
print(f'\nSpectrograms : {n_spec} files')
print(f'Labels       : {n_label} files')

Mounted at /content/drive
Project root exists : True
src/ exists         : True
spectrograms/ exists: True
labels/ exists      : True

Spectrograms : 3163 files
Labels       : 3163 files


## Cell 2 — Paths và imports

In [4]:
import numpy as np
import pandas as pd
import json
import csv
import time
import torch
import torch.nn as nn
from tqdm import tqdm

# Paths
SPEC_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'spectrograms'
LABEL_DIR  = PROJECT_ROOT / 'data' / 'processed' / 'labels'
META_DIR   = PROJECT_ROOT / 'data' / 'metadata'
MODELS_DIR = PROJECT_ROOT / 'models' / 'rnn'
LOGS_DIR   = PROJECT_ROOT / 'experiments' / 'logs'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

print('Paths OK')

Device: cuda
Paths OK


## Cell 3 — Load metadata

In [4]:
patients_df   = pd.read_csv(META_DIR / 'patients.csv')
recordings_df = pd.read_csv(META_DIR / 'recordings.csv')

# Đảm bảo recording_id đúng (lấy từ stem wav_path)
if 'recording_id' not in recordings_df.columns:
    recordings_df['recording_id'] = recordings_df['wav_path'].apply(
        lambda p: Path(p).stem
    )

print(f'patients_df  : {patients_df.shape}')
print(f'recordings_df: {recordings_df.shape}')
print(recordings_df[['patient_id', 'location', 'recording_id']].head(3))

patients_df  : (942, 24)
recordings_df: (3163, 10)
   patient_id location recording_id
0        2530       AV      2530_AV
1        2530       PV      2530_PV
2        2530       TV      2530_TV


## Cell 4 — Load CV splits

In [5]:
with open(META_DIR / 'cv_splits.json') as f:
    cv_splits = json.load(f)

print('CV splits loaded:')
for fold_name, fold_data in cv_splits.items():
    print(f'  {fold_name}: {len(fold_data["train_recordings"])} train, '
          f'{len(fold_data["val_recordings"])} val recordings')

CV splits loaded:
  fold_0: 2553 train, 610 val recordings
  fold_1: 2516 train, 647 val recordings
  fold_2: 2519 train, 644 val recordings
  fold_3: 2535 train, 628 val recordings
  fold_4: 2529 train, 634 val recordings


## Cell 5 — Load toàn bộ dataset vào RAM (~2-3 phút)

In [7]:
# Cell 5a — Copy data từ Drive vào Colab local storage (~3-5 phút, chỉ 1 lần)
import shutil

LOCAL_SPEC_DIR  = Path('/content/spectrograms')
LOCAL_LABEL_DIR = Path('/content/labels')

if not LOCAL_SPEC_DIR.exists():
    print("Copying spectrograms từ Drive → /content/ ...")
    shutil.copytree(SPEC_DIR, LOCAL_SPEC_DIR)
    print(f"Done: {len(list(LOCAL_SPEC_DIR.glob('*.npy')))} files")
else:
    print(f"Đã có: {len(list(LOCAL_SPEC_DIR.glob('*.npy')))} spectrograms")

if not LOCAL_LABEL_DIR.exists():
    print("Copying labels từ Drive → /content/ ...")
    shutil.copytree(LABEL_DIR, LOCAL_LABEL_DIR)
    print(f"Done: {len(list(LOCAL_LABEL_DIR.glob('*.npy')))} files")
else:
    print(f"Đã có: {len(list(LOCAL_LABEL_DIR.glob('*.npy')))} labels")

Copying spectrograms từ Drive → /content/ ...
Done: 3163 files
Copying labels từ Drive → /content/ ...
Done: 3163 files


In [8]:
# Cell 5b — Load vào RAM từ local storage (nhanh như máy local ~2-3 phút)
from src.data.loader import load_dataset_to_ram, create_dataloader

spec_cache, label_cache, lengths_dict = load_dataset_to_ram(
    recordings_df['recording_id'].tolist(),
    LOCAL_SPEC_DIR,   # ← đọc từ /content/ không phải Drive
    LOCAL_LABEL_DIR
)

Loading dataset to RAM: 100%|██████████| 3163/3163 [00:02<00:00, 1470.27it/s]

RAM used: ~1191 MB — 3163 recordings cached


## Cell 6 — Tính class weights

In [9]:
# Đếm frame theo trạng thái trên toàn bộ dataset
state_names = ['S1', 'Systole', 'S2', 'Diastole', 'Murmur']
frame_counts = [0] * 5

for labels_arr in tqdm(label_cache.values(), desc='Counting frames'):
    for state in range(5):
        frame_counts[state] += int(np.sum(labels_arr == state))

n_classes      = 5
total_labelled = sum(frame_counts)
weights        = [total_labelled / (n_classes * c) for c in frame_counts]
min_w          = min(weights)
weights        = [w / min_w for w in weights]
weights_tensor = torch.FloatTensor(weights).to(DEVICE)

print(f'{'State':>12}  {'Frames':>10}  {'%':>6}  {'Weight':>8}')
print('-' * 45)
for name, count, w in zip(state_names, frame_counts, weights):
    pct = 100 * count / total_labelled
    print(f'{name:>12}  {count:>10,}  {pct:>5.1f}%  {w:>8.3f}')

Counting frames: 100%|██████████| 3163/3163 [00:00<00:00, 33064.95it/s]


       State      Frames       %    Weight
---------------------------------------------
          S1     385,582   20.3%     1.922
     Systole     378,660   19.9%     1.957
          S2     337,989   17.8%     2.193
    Diastole     741,055   39.0%     1.000
      Murmur      55,591    2.9%    13.330


## Cell 7 — Config huấn luyện

In [10]:
CONFIG = {
    'lr':          1e-3,
    'batch_size':  32,    # GPU T4 có thể dùng batch lớn hơn CPU
    'max_epochs':  100,
    'patience':    10,
    'grad_clip':   1.0,
    'seed':        42,
    'num_workers': 2,     # Colab Linux hỗ trợ multi-worker
}

print('Config:')
for k, v in CONFIG.items():
    print(f'  {k:>12}: {v}')

Config:
            lr: 0.001
    batch_size: 32
    max_epochs: 100
      patience: 10
     grad_clip: 1.0
          seed: 42
   num_workers: 2


## Cell 8 — Định nghĩa hàm train/val một epoch

In [11]:
def run_epoch(model, loader, criterion, optimiser=None,
              is_train=True, fold_idx=0, epoch=0):
    """Chạy 1 epoch train hoặc val. Trả về loss trung bình."""
    model.train() if is_train else model.eval()
    total_loss = 0.0
    n_batches  = 0
    mode       = 'train' if is_train else 'val'

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader,
                    desc=f'  Fold {fold_idx} Ep {epoch:>3} {mode}',
                    leave=False, ncols=90)
        for batch in pbar:
            # Chuyển sang GPU
            features = batch['features'].to(DEVICE)
            labels   = batch['labels'].to(DEVICE)
            lengths  = batch['lengths']

            logits = model(features, lengths)          # (B, T_max, 5)
            loss   = criterion(logits.permute(0,2,1),  # (B, 5, T_max)
                               labels)

            if is_train:
                optimiser.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), CONFIG['grad_clip'])
                optimiser.step()

            total_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / n_batches if n_batches > 0 else 0.0

## Cell 9 — Vòng lặp huấn luyện 5-fold (~5-7 giờ trên T4)

In [12]:
from src.models.rnn import build_model

torch.manual_seed(CONFIG['seed'])

for fold_idx in range(5):
    fold_name = f'fold_{fold_idx}'
    print(f"\n{'='*60}")
    print(f'  {fold_name.upper()}  ({fold_idx+1}/5)')
    print(f"{'='*60}")

    # DataLoaders — đọc từ RAM cache
    train_ids = cv_splits[fold_name]['train_recordings']
    val_ids   = cv_splits[fold_name]['val_recordings']

    train_loader = create_dataloader(
        train_ids, SPEC_DIR, LABEL_DIR,
        batch_size=CONFIG['batch_size'], shuffle=True,
        num_workers=CONFIG['num_workers'],
        spec_cache=spec_cache, label_cache=label_cache,
        lengths_dict=lengths_dict
    )
    val_loader = create_dataloader(
        val_ids, SPEC_DIR, LABEL_DIR,
        batch_size=CONFIG['batch_size'], shuffle=False,
        num_workers=CONFIG['num_workers'],
        spec_cache=spec_cache, label_cache=label_cache,
        lengths_dict=lengths_dict
    )
    print(f'  Train: {len(train_ids)} recordings, {len(train_loader)} batches')
    print(f'  Val  : {len(val_ids)} recordings, {len(val_loader)} batches')

    # Model + optimiser + loss (fresh mỗi fold)
    torch.manual_seed(CONFIG['seed'])
    model     = build_model(seed=CONFIG['seed']).to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.CrossEntropyLoss(
        weight=weights_tensor,
        ignore_index=-1
    )

    best_val_loss  = float('inf')
    patience_count = 0
    log_rows       = []
    fold_start     = time.time()

    for epoch in range(1, CONFIG['max_epochs'] + 1):
        ep_start   = time.time()
        train_loss = run_epoch(model, train_loader, criterion,
                               optimiser, is_train=True,
                               fold_idx=fold_idx, epoch=epoch)
        val_loss   = run_epoch(model, val_loader, criterion,
                               is_train=False,
                               fold_idx=fold_idx, epoch=epoch)
        ep_secs    = time.time() - ep_start

        log_rows.append({
            'epoch':      epoch,
            'train_loss': round(train_loss, 6),
            'val_loss':   round(val_loss,   6),
        })

        # In kết quả mỗi epoch
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:>3}  '
                  f'train={train_loss:.4f}  '
                  f'val={val_loss:.4f}  '
                  f'({ep_secs:.0f}s)')

        # Lưu checkpoint nếu val loss tốt hơn
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            torch.save({
                'epoch':             epoch,
                'model_state_dict':  model.state_dict(),
                'val_loss':          val_loss,
                'config':            CONFIG,
                'fold':              fold_idx,
            }, MODELS_DIR / f'{fold_name}_best.pt')
        else:
            patience_count += 1

        # Early stopping
        if patience_count >= CONFIG['patience']:
            print(f'  Early stopping tại epoch {epoch} '
                  f'(no improvement for {CONFIG["patience"]} epochs)')
            break

    # Lưu loss log
    log_path = LOGS_DIR / f'{fold_name}_loss.csv'
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss'])
        writer.writeheader()
        writer.writerows(log_rows)

    fold_mins = (time.time() - fold_start) / 60
    print(f'\n  ✅ {fold_name} done — '
          f'best val loss: {best_val_loss:.4f} '
          f'({fold_mins:.1f} min)')
    print(f'  Checkpoint: {MODELS_DIR/f"{fold_name}_best.pt"}')

print(f"\n{'='*60}")
print('✅ Huấn luyện hoàn tất — 5/5 fold')


  FOLD_0  (1/5)
  Train: 2553 recordings, 80 batches
  Val  : 610 recordings, 20 batches


  Epoch   1  train=0.9611  val=0.5401  (29s)


  Epoch   5  train=0.4433  val=0.3971  (28s)


  Epoch  10  train=0.3870  val=0.3593  (27s)


  Epoch  15  train=0.3532  val=0.3537  (28s)


...

  Epoch  10  train=0.3761  val=0.4161  (28s)


  Epoch  15  train=0.3469  val=0.4187  (28s)


  Epoch  20  train=0.3205  val=0.4240  (28s)


  Epoch  25  train=0.2817  val=0.4781  (28s)


  Early stopping tại epoch 28 (no improvement for 10 epochs)

  ✅ fold_4 done — best val loss: 0.3958 (13.0 min)
  Checkpoint: /content/drive/MyDrive/heart-murmur/models/rnn/fold_4_best.pt

✅ Huấn luyện hoàn tất — 5/5 fold


## Cell 10 — Kiểm tra checkpoint đã lưu

In [ ]:
print('Checkpoint files:')
for pt_file in sorted(MODELS_DIR.glob('*.pt')):
    ckpt = torch.load(pt_file, map_location='cpu')
    print(f'  {pt_file.name}: '
          f'epoch={ckpt["epoch"]}, '
          f'val_loss={ckpt["val_loss"]:.4f}')

print('\nLog files:')
for csv_file in sorted(LOGS_DIR.glob('*.csv')):
    df = pd.read_csv(csv_file)
    best_ep  = df.loc[df['val_loss'].idxmin()]
    print(f'  {csv_file.name}: '
          f'{len(df)} epochs, '
          f'best val={best_ep["val_loss"]:.4f} at epoch {int(best_ep["epoch"])}')

## Cell 11 — Download checkpoint về máy local

Sau khi train xong, copy folder về máy:
- `MyDrive/heart-murmur/models/rnn/` → `D:\...\heart-murmur-detection\models\rnn\`
- `MyDrive/heart-murmur/experiments/logs/` → `D:\...\heart-murmur-detection\experiments\logs\`

Hoặc dùng cell dưới để zip và download trực tiếp:

In [5]:
import shutil
from google.colab import files

# Zip toàn bộ models + logs
zip_path = '/content/rnn_checkpoints.zip'
shutil.make_archive(
    '/content/rnn_checkpoints',
    'zip',
    PROJECT_ROOT,
    'models'
)

# Download về máy
files.download(zip_path)
print('Download started — kiểm tra folder Downloads')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started — kiểm tra folder Downloads
